In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
from functools import reduce
from typing import Sequence

import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv("../../results/synthetic_data/hier-projmix_results_raw_long.csv")
df_ari = df[df["Metric"].eq("full ari")].copy()

In [3]:
df_ari

,Seed,Model,Model Type,Metric,Sample,Noise,Experiment,Value,Dimension,Separability,Dependence,K,L
68,0,"(2,1)-Two-layer MoM",mom,full ari,in sample,no noise,2d-low-strong,0.00590,2,low,strong,2,1
74,0,"(2,2)-Two-layer MoM",mom,full ari,in sample,no noise,2d-low-strong,0.09171,2,low,strong,2,2
80,0,"(2,3)-Two-layer MoM",mom,full ari,in sample,no noise,2d-low-strong,0.06692,2,low,strong,2,3
86,0,"(2,1)-Isolated Two-layer MoM",isomom,full ari,in sample,no noise,2d-low-strong,0.00311,2,low,strong,2,1
92,0,"(2,2)-Isolated Two-layer MoM",isomom,full ari,in sample,no noise,2d-low-strong,0.09096,2,low,strong,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
895151,99,"(4,2)-Two-layer MoM",mom,full ari,out of sample,with noise,3d-high-ind,0.25701,3,high,ind,4,2
895157,99,"(4,3)-Two-layer MoM",mom,full ari,out of sample,with noise,3d-high-ind,0.22302,3,high,ind,4,3
895163,99,"(4,1)-Isolated Two-layer MoM",isomom,full ari,out of sample,with noise,3d-high-ind,0.19106,3,high,ind,4,1
895169,99,"(4,2)-Isolated Two-layer MoM",isomom,full ari,out of sample,with noise,3d-high-ind,0.26570,3,high,ind,4,2


In [4]:
df.columns

Index(['Seed', 'Model', 'Model Type', 'Metric', 'Sample', 'Noise',
       'Experiment', 'Value', 'Dimension', 'Separability', 'Dependence', 'K',
       'L'],
      dtype='object')

In [5]:
df.head()

,Seed,Model,Model Type,Metric,Sample,Noise,Experiment,Value,Dimension,Separability,Dependence,K,L
0,0,all,all,sample_cross_cov,training,no noise,2d-low-strong,0.04,2,low,strong,0,0
1,0,2-Cylindrical Mixture,cylmix,training_time,training,no noise,2d-low-strong,444.68,2,low,strong,2,0
2,0,2-Cylindrical Mixture,cylmix,em_iters,training,no noise,2d-low-strong,17.00,2,low,strong,2,0
3,0,2-Independent Cylindrical Mixture,indcylmix,training_time,training,no noise,2d-low-strong,408.65,2,low,strong,2,0
4,0,2-Independent Cylindrical Mixture,indcylmix,em_iters,training,no noise,2d-low-strong,18.00,2,low,strong,2,0


In [6]:
def as_list(value):
    return [value] if isinstance(value, str) else list(value)


def metric_spec(metric, sample, prefix, summary, lower_is_better=True):
    return {
        "metric": metric,
        "sample": sample,
        "prefix": prefix,
        "summary": summary,
        "lower_is_better": lower_is_better,
    }


COMPARISON_SPECS = [
    metric_spec("avg_ll", "out of sample", "avg_ll", "diff"),
    metric_spec("bic", "in sample", "bic", "win", lower_is_better=True),
    metric_spec("aic", "in sample", "aic", "win", lower_is_better=True),
    metric_spec("avg_ll", "out of sample", "hll", "win", lower_is_better=False),
    metric_spec("ari", "out of sample", "ari", "diff"),
    metric_spec("training_time", "training", "time", "ratio"),
    metric_spec("em_iters", "training", "em_iters", "ratio"),
]

AVG_LL_COLUMNS = {"avg_ll_mean", "avg_ll_std"}
PERCENT_COLUMNS = {"ari_mean"}


def is_win_rate_column(column):
    return isinstance(column, str) and column.endswith("_win_rate")


def is_time_or_em_iters_column(column):
    return isinstance(column, str) and ("time" in column or "em_iters" in column)


def metric_decimals(column, default=None):
    if column in AVG_LL_COLUMNS:
        return 5
    if column in PERCENT_COLUMNS or is_win_rate_column(column):
        return 2
    if is_time_or_em_iters_column(column):
        return 3
    return default


def round_metric_columns(table, default=None, skip_columns=()):
    rounded = table.copy()
    skip_columns = set(skip_columns)

    for column in rounded.columns:
        if column in skip_columns or not pd.api.types.is_numeric_dtype(rounded[column]):
            continue

        decimals = metric_decimals(column, default=default)
        if decimals is not None:
            rounded[column] = rounded[column].round(decimals)

    return rounded


def format_metric_table(table):
    formatted = table.copy()
    percent_columns = [
        column
        for column in formatted.columns
        if (column in PERCENT_COLUMNS or is_win_rate_column(column))
        and pd.api.types.is_numeric_dtype(formatted[column])
    ]

    if percent_columns:
        formatted[percent_columns] = formatted[percent_columns].mul(100)

    return round_metric_columns(formatted)


def format_metric_values(table, source_column):
    formatted = table.copy()

    if source_column in PERCENT_COLUMNS or is_win_rate_column(source_column):
        formatted = formatted.mul(100)

    decimals = metric_decimals(source_column)
    if decimals is not None:
        formatted = formatted.round(decimals)

    return formatted


def metric_view(df, metric, sample, model1, model2, model_col="Model Type"):
    samples = as_list(sample)

    metric_df = df[
        (df["Metric"] == metric)
        & (df["Sample"].isin(samples))
        & (df[model_col].isin([model1, model2]))
    ]

    return metric_df


def paired_model_values(metric_df, model1, model2, match_cols, model_col="Model Type"):
    paired = metric_df.pivot_table(
        index=match_cols,
        columns=model_col,
        values="Value",
        aggfunc="mean",
    )

    missing = [model for model in [model1, model2] if model not in paired.columns]
    if missing:
        available = paired.columns.to_list()
        raise KeyError(
            f"Missing paired columns for {model_col}: {missing}. "
            f"Available columns: {available}"
        )

    paired = paired.dropna(subset=[model1, model2]).reset_index()
    paired.columns.name = None
    paired["diff"] = paired[model1] - paired[model2]
    paired["ratio"] = paired[model1] / paired[model2]

    return paired


def summarize_diff(
    metric_df,
    model1,
    model2,
    match_cols,
    prefix,
    groupby=None,
    model_col="Model Type",
):
    groupby = ["Dependence"] if groupby is None else groupby

    paired = paired_model_values(
        metric_df, model1, model2, match_cols, model_col=model_col
    )

    return (
        paired
        .groupby(groupby, as_index=False)
        .agg(
            **{
                f"{prefix}_mean": ("diff", "mean"),
                f"{prefix}_std": ("diff", "std"),
            }
        )
    )


def summarize_ratio(
    metric_df,
    model1,
    model2,
    match_cols,
    prefix,
    groupby=None,
    model_col="Model Type",
):
    groupby = ["Dependence"] if groupby is None else groupby

    paired = paired_model_values(
        metric_df, model1, model2, match_cols, model_col=model_col
    )

    return (
        paired
        .groupby(groupby, as_index=False)
        .agg(
            **{
                f"{prefix}_mean": ("ratio", "mean"),
                f"{prefix}_std": ("ratio", "std"),
            }
        )
    )


def summarize_wins(
    metric_df,
    model1,
    model2,
    match_cols,
    prefix,
    groupby=None,
    lower_is_better=True,
    model_col="Model Type",
):
    groupby = ["Dependence"] if groupby is None else groupby

    paired = paired_model_values(
        metric_df, model1, model2, match_cols, model_col=model_col
    )

    if lower_is_better:
        paired["win"] = paired[model1] < paired[model2]
    else:
        paired["win"] = paired[model1] > paired[model2]

    return (
        paired
        .groupby(groupby, as_index=False)
        .agg(
            **{
                f"{prefix}_win_rate": ("win", "mean"),
                f"{prefix}_win_count": ("win", "sum"),
            }
        )
    )


def summarize_metric(df, spec, model1, model2, match_cols, groupby, model_col="Model Type"):
    metric_df = metric_view(
        df,
        spec["metric"],
        spec["sample"],
        model1,
        model2,
        model_col=model_col,
    )
    summary = spec["summary"]

    if summary == "diff":
        return summarize_diff(
            metric_df,
            model1,
            model2,
            match_cols,
            spec["prefix"],
            groupby,
            model_col=model_col,
        )
    if summary == "ratio":
        return summarize_ratio(
            metric_df,
            model1,
            model2,
            match_cols,
            spec["prefix"],
            groupby,
            model_col=model_col,
        )
    if summary == "win":
        return summarize_wins(
            metric_df,
            model1,
            model2,
            match_cols,
            spec["prefix"],
            groupby,
            lower_is_better=spec.get("lower_is_better", True),
            model_col=model_col,
        )

    raise ValueError(f"Unknown summary type: {summary}")


def merge_table_parts(table_parts, groupby):
    return reduce(
        lambda left, right: left.merge(right, on=as_list(groupby), how="outer"),
        table_parts,
    )


def comparison_table(
    df,
    model1,
    model2,
    match_cols,
    groupby,
    specs,
    model_col="Model Type",
    format_output=True,
):
    table_parts = [
        summarize_metric(
            df,
            spec,
            model1,
            model2,
            match_cols,
            groupby,
            model_col=model_col,
        )
        for spec in specs
    ]

    table = merge_table_parts(table_parts, groupby)
    if format_output:
        table = format_metric_table(table)

    return table


def metric_win_rate_pivot(
    df,
    metric,
    sample,
    model1,
    model2,
    match_cols,
    index,
    pivot_column,
    model_col="Model Type",
    lower_is_better=True,
    index_order=None,
    column_order=None,
    scale=100,
    decimals=2,
):
    metric_df = metric_view(
        df, metric, sample, model1, model2, model_col=model_col
    )
    paired = paired_model_values(
        metric_df, model1, model2, match_cols, model_col=model_col
    )

    if lower_is_better:
        paired["win"] = paired[model1] < paired[model2]
    else:
        paired["win"] = paired[model1] > paired[model2]

    table = (
        paired
        .groupby([index, pivot_column], as_index=False)
        .agg(win_rate=("win", "mean"))
        .pivot(index=index, columns=pivot_column, values="win_rate")
    )

    if index_order is not None or column_order is not None:
        table = table.reindex(index=index_order, columns=column_order)
    if scale is not None:
        table = table.mul(scale)
    if decimals is not None:
        table = table.round(decimals)

    return table.reset_index()


def metric_diff_pivot(
    df,
    metric,
    sample,
    model1,
    model2,
    match_cols,
    index,
    pivot_column,
    model_col="Model Type",
    value_name="diff_mean",
    aggfunc="mean",
    index_order=None,
    column_order=None,
):
    metric_df = metric_view(
        df, metric, sample, model1, model2, model_col=model_col
    )
    paired = paired_model_values(
        metric_df, model1, model2, match_cols, model_col=model_col
    )

    table = (
        paired
        .groupby([index, pivot_column], as_index=False)
        .agg(**{value_name: ("diff", aggfunc)})
        .pivot(index=index, columns=pivot_column, values=value_name)
    )

    if index_order is not None or column_order is not None:
        table = table.reindex(index=index_order, columns=column_order)

    table = format_metric_values(table, value_name)

    return table.reset_index()


def ari_recovery_table(source_df, scale=100, decimals=2):
    ari_table_df = source_df[
        (source_df["Metric"] == "ari")
        & (source_df["Sample"] == "out of sample")
        & (source_df["K"] == 3)
    ].copy()

    ari_table_df["Model Label"] = np.select(
        [
            ari_table_df["Model Type"].eq("cylmix"),
            ari_table_df["Model Type"].eq("indcylmix"),
            ari_table_df["Model Type"].eq("mom") & ari_table_df["L"].eq(2),
            ari_table_df["Model Type"].eq("isomom") & ari_table_df["L"].eq(2),
        ],
        [
            "Cylindrical",
            "Independent cylindrical",
            "Full MoM",
            "Isolated MoM",
        ],
        default=None,
    )

    table = (
        ari_table_df
        .dropna(subset=["Model Label"])
        .groupby(["Separability", "Dependence", "Model Label"], as_index=False)
        .agg(ari=("Value", "mean"))
        .pivot_table(
            index=["Separability", "Dependence"],
            columns="Model Label",
            values="ari",
            aggfunc="mean",
        )
        .reindex(
            index=pd.MultiIndex.from_product(
                [["high", "low"], ["ind", "weak", "strong"]],
                names=["Separability", "Dependence"],
            ),
            columns=[
                "Cylindrical",
                "Independent cylindrical",
                "Full MoM",
                "Isolated MoM",
            ],
        )
        .rename(
            index={
                "high": "High",
                "low": "Low",
                "ind": "Conditionally independent",
                "weak": "Weak",
                "strong": "Strong",
            }
        )
    )

    if scale is not None:
        table = table.mul(scale)
    if decimals is not None:
        table = table.round(decimals)

    table = table.reset_index()
    table.columns.name = None

    return table


### Can the Two-layer Model recover the clusters?

In [7]:
ari_table_df = df_ari[
    (df_ari["Sample"] == "out of sample")
].copy()


table_ari_full = (
    ari_table_df
    .groupby(["Separability", "Dependence", "Model"], as_index=False)
    .agg(ari=("Value", "mean"))
    .pivot_table(
        index=["Separability", "Dependence"],
        columns="Model",
        values="ari",
        aggfunc="mean",
    )

)
table_ari_full = table_ari_full.mul(100).round(2).reset_index()
table_ari_full.columns.name = None
table_ari_full[['Dependence','Separability','(3,2)-Two-layer MoM','(3,2)-Isolated Two-layer MoM']]

,Dependence,Separability,"(3,2)-Two-layer MoM","(3,2)-Isolated Two-layer MoM"
0,ind,high,25.32,26.90
1,strong,high,45.31,32.57
2,weak,high,31.80,26.06
3,ind,low,1.95,1.50
4,strong,low,8.17,7.43
5,weak,low,2.61,2.19


In [8]:
table_ari_full

,Separability,Dependence,"(2,1)-Isolated Two-layer MoM","(2,1)-Two-layer MoM","(2,2)-Isolated Two-layer MoM","(2,2)-Two-layer MoM","(2,3)-Isolated Two-layer MoM","(2,3)-Two-layer MoM","(3,1)-Isolated Two-layer MoM","(3,1)-Two-layer MoM","(3,2)-Isolated Two-layer MoM","(3,2)-Two-layer MoM","(3,3)-Isolated Two-layer MoM","(3,3)-Two-layer MoM","(4,1)-Isolated Two-layer MoM","(4,1)-Two-layer MoM","(4,2)-Isolated Two-layer MoM","(4,2)-Two-layer MoM","(4,3)-Isolated Two-layer MoM","(4,3)-Two-layer MoM"
0,high,ind,16.75,16.76,29.24,29.18,24.17,23.97,17.41,18.58,26.90,25.32,21.69,21.02,16.68,17.46,23.60,22.62,18.66,18.44
1,high,strong,17.90,17.42,34.38,31.59,35.43,38.26,21.28,39.42,32.57,45.31,32.51,45.58,21.99,39.52,29.54,42.43,29.04,38.72
2,high,weak,16.93,12.66,27.79,26.97,25.62,24.92,17.96,24.95,26.06,31.80,23.20,27.92,17.51,24.64,23.20,28.24,20.30,23.36
3,low,ind,0.18,0.24,1.35,1.70,1.81,1.95,0.67,0.74,1.50,1.95,1.80,1.92,0.70,0.90,1.35,1.80,1.56,1.68
4,low,strong,0.21,4.47,8.60,9.28,7.79,8.34,0.75,6.27,7.43,8.17,6.53,7.00,0.81,6.21,6.30,6.95,5.44,5.81
5,low,weak,0.19,0.33,2.21,2.46,2.53,2.66,0.67,1.14,2.19,2.61,2.38,2.48,0.71,1.39,1.93,2.35,2.04,2.14


### Does the cylindrical cross-dependence parameter matter?

In [9]:
model1 = "cylmix"
model2 = "indcylmix"

match_cols_cyl = [
    "Seed",
    "K",
    "Noise",
    "Experiment",
    "Dimension",
    "Separability",
    "Dependence",
    "Sample",
]

groupby_cyl = ['Dependence']

table_cyl_dependence = comparison_table(
    df,
    model1,
    model2,
    match_cols_cyl,
    groupby_cyl,
    COMPARISON_SPECS,
)


In [10]:
table_cyl_dependence[['Dependence', 'avg_ll_mean', 'avg_ll_std', 'bic_win_rate', 'aic_win_rate', 'ari_mean', 'time_mean', 'em_iters_mean']]

,Dependence,avg_ll_mean,avg_ll_std,bic_win_rate,aic_win_rate,ari_mean,time_mean,em_iters_mean
0,ind,-0.00099,0.00279,0.12,13.35,0.17,1.193,1.214
1,strong,0.00452,0.01542,11.77,34.70,0.01,1.186,1.244
2,weak,0.00060,0.00497,7.68,31.08,0.66,1.243,1.437


In [11]:
table_cyl_bic = metric_win_rate_pivot(
    df,
    "bic",
    "in sample",
    model1,
    model2,
    match_cols_cyl,
    index="Dependence",
    pivot_column="K",
    index_order=["ind", "weak", "strong"],
    column_order=[2, 3, 4],
)

table_cyl_bic


K,Dependence,2,3,4
0,ind,0.0,0.25,0.38
1,weak,20.5,16.62,1.25
2,strong,50.0,0.00,5.12


In [12]:
table_cyl_ari = metric_diff_pivot(
    df,
    "ari",
    ["in sample", "out of sample"],
    model1,
    model2,
    match_cols_cyl,
    index="Dependence",
    pivot_column="K",
    value_name="ari_mean",
    index_order=["ind", "weak", "strong"],
    column_order=[2, 3, 4],
)

table_cyl_ari


K,Dependence,2,3,4
0,ind,0.08,1.67,-0.24
1,weak,3.89,1.83,-0.56
2,strong,2.18,-1.17,-0.50


### Does the directional block contain useful information for determining the Euclidean latent class?

In [13]:
model1 = "mom"
model2 = "isomom"

match_cols_mom = [
    "Seed",
    "K",
    "L",
    "Noise",
    "Experiment",
    "Dimension",
    "Separability",
    "Dependence",
    "Sample",
]

groupby_mom = ["Dependence", "L"]

table_mom_dependence = comparison_table(
    df,
    model1,
    model2,
    match_cols_mom,
    groupby_mom,
    COMPARISON_SPECS,
)

table_mom_dependence[['L', 'Dependence', 'avg_ll_mean', 'avg_ll_std', 'bic_win_rate', 'aic_win_rate', 'ari_mean', 'time_mean', 'em_iters_mean']]


,L,Dependence,avg_ll_mean,avg_ll_std,bic_win_rate,aic_win_rate,ari_mean,time_mean,em_iters_mean
0,1,ind,0.00804,0.01825,99.42,99.42,-2.11,2.220,0.803
1,2,ind,0.00296,0.00541,99.33,99.33,-0.71,1.990,0.617
2,3,ind,0.00226,0.00492,99.08,99.08,-0.14,1.802,0.401
3,1,strong,0.08554,0.08362,99.25,99.25,11.36,2.412,0.955
4,2,strong,0.04168,0.04469,100.00,100.00,9.59,2.644,1.109
5,3,strong,0.03468,0.03546,99.92,99.92,9.66,2.219,0.635
6,1,weak,0.01925,0.02700,99.75,99.75,-1.69,2.718,1.165
7,2,weak,0.00943,0.00953,99.62,99.62,2.13,2.298,0.828
8,3,weak,0.00720,0.00883,99.75,99.75,2.16,2.024,0.514


In [14]:
print(round(table_mom_dependence[table_mom_dependence["Dependence"] == "ind"]["hll_win_count"].sum() / 4800.0 * 100, 2))
print(round(table_mom_dependence[table_mom_dependence["Dependence"] == "weak"]["hll_win_count"].sum() / 4800.0 * 100, 2))
print(round(table_mom_dependence[table_mom_dependence["Dependence"] == "strong"]["hll_win_count"].sum() / 4800.0 * 100, 2))


94.31
126.21
140.31


In [15]:
model1 = "mom"
model2 = "isomom"

match_cols_mom = [
    "Seed",
    "K",
    "L",
    "Noise",
    "Experiment",
    "Dimension",
    "Separability",
    "Dependence",
    "Sample",
]

groupby_mom = ["Separability", "L"]

table_mom_separability = comparison_table(
    df,
    model1,
    model2,
    match_cols_mom,
    groupby_mom,
    COMPARISON_SPECS,
)

table_mom_separability[['L', 'Separability', 'avg_ll_mean', 'avg_ll_std', 'bic_win_rate', 'aic_win_rate', 'ari_mean', 'time_mean', 'em_iters_mean']]


,L,Separability,avg_ll_mean,avg_ll_std,bic_win_rate,aic_win_rate,ari_mean,time_mean,em_iters_mean
0,1,high,0.06691,0.07590,99.72,99.72,2.73,2.479,1.023
1,2,high,0.03277,0.03913,100.00,100.00,6.73,2.683,1.116
2,3,high,0.02778,0.03102,99.67,99.67,7.24,2.338,0.682
3,1,low,0.00830,0.01515,99.22,99.22,2.31,2.421,0.925
4,2,low,0.00327,0.00427,99.31,99.31,0.62,1.939,0.587
5,3,low,0.00165,0.00312,99.50,99.50,0.55,1.692,0.352


In [16]:
model1 = "mom"
model2 = "isomom"

match_cols_mom = [
    "Seed",
    "K",
    "L",
    "Noise",
    "Experiment",
    "Dimension",
    "Separability",
    "Dependence",
    "Sample",
]

groupby_mom = ["Dependence", "Separability", "L"]

table_mom_separability2 = comparison_table(
    df,
    model1,
    model2,
    match_cols_mom,
    groupby_mom,
    COMPARISON_SPECS,
)

table_mom_separability2[table_mom_separability2.Dependence.eq("ind")][
    ['L', 'Separability', 'avg_ll_mean']
]

,L,Separability,avg_ll_mean
0,1,high,0.01322
1,2,high,0.00425
2,3,high,0.00363
3,1,low,0.00285
4,2,low,0.00166
5,3,low,0.00089


### How does the cylindrical mixture model compare against the hierarchical mixture model?

In [17]:
model1 = "3-Cylindrical Mixture"
model2 = "(3,2)-Two-layer MoM"

match_cols_vs = [
    "Seed",
    "Noise",
    "Experiment",
    "Dimension",
    "Separability",
    "Dependence",
    "Sample",
]

groupby_vs = ["Dependence", "Separability"]

table_vs = comparison_table(
    df,
    model1,
    model2,
    match_cols_vs,
    groupby_vs,
    COMPARISON_SPECS,
    model_col="Model",
)

table_vs[['Dependence','Separability','avg_ll_mean','avg_ll_std','bic_win_rate','aic_win_rate','hll_win_rate','ari_mean', 'time_mean','em_iters_mean']]


,Dependence,Separability,avg_ll_mean,avg_ll_std,bic_win_rate,aic_win_rate,hll_win_rate,ari_mean,time_mean,em_iters_mean
0,ind,high,-0.01385,0.01332,0.50,7.25,13.25,-17.12,0.398,1.089
1,ind,low,-0.00014,0.00308,23.25,59.50,50.50,-0.80,0.409,1.056
2,strong,high,-0.02152,0.01931,1.25,7.25,8.25,-2.27,0.201,0.247
3,strong,low,-0.00064,0.00332,24.25,52.75,49.50,11.79,0.345,0.819
4,weak,high,-0.01154,0.01037,4.00,5.25,8.00,-17.79,0.418,1.125
5,weak,low,-0.00014,0.00319,22.00,57.75,53.75,0.72,0.397,0.996


### Clustering recovery

In [18]:
ari_table = ari_recovery_table(df)

ari_table

,Separability,Dependence,Cylindrical,Independent cylindrical,Full MoM,Isolated MoM
0,High,Conditionally independent,10.66,7.47,27.78,30.28
1,High,Weak,22.03,18.16,39.82,31.06
2,High,Strong,60.67,61.43,62.94,35.41
3,Low,Conditionally independent,0.26,0.07,1.06,1.14
4,Low,Weak,1.91,2.10,1.19,1.15
5,Low,Strong,15.19,16.78,3.40,1.30


### With-noise-only tables

In [19]:
from IPython.display import display

with_noise_df = df[df["Noise"].eq("with noise")].copy()

table_cyl_dependence_with_noise = comparison_table(
    with_noise_df,
    "cylmix",
    "indcylmix",
    match_cols_cyl,
    ["Dependence"],
    COMPARISON_SPECS,
)

table_mom_dependence_with_noise = comparison_table(
    with_noise_df,
    "mom",
    "isomom",
    match_cols_mom,
    ["Dependence", "L"],
    COMPARISON_SPECS,
)

table_mom_separability_with_noise = comparison_table(
    with_noise_df,
    "mom",
    "isomom",
    match_cols_mom,
    ["Separability", "L"],
    COMPARISON_SPECS,
)

table_vs_with_noise = comparison_table(
    with_noise_df,
    "3-Cylindrical Mixture",
    "(3,2)-Two-layer MoM",
    match_cols_vs,
    ["Dependence", "Separability"],
    COMPARISON_SPECS,
    model_col="Model",
)

ari_table_with_noise = ari_recovery_table(with_noise_df)

with_noise_tables = {
    "table_cyl_dependence": table_cyl_dependence_with_noise[["Dependence", "avg_ll_mean", "avg_ll_std", "bic_win_rate", "aic_win_rate", "ari_mean", "time_mean", "em_iters_mean"]],
    "table_mom_dependence": table_mom_dependence_with_noise[["L", "Dependence", "avg_ll_mean", "avg_ll_std", "bic_win_rate", "aic_win_rate", "time_mean", "em_iters_mean"]],
    "table_mom_separability": table_mom_separability_with_noise[["L", "Separability", "avg_ll_mean", "avg_ll_std", "bic_win_rate", "aic_win_rate", "time_mean", "em_iters_mean"]],
    "table_vs": table_vs_with_noise[["Dependence", "Separability", "avg_ll_mean", "avg_ll_std", "bic_win_rate", "aic_win_rate", "hll_win_rate", "time_mean", "em_iters_mean"]],
    "ari_table": ari_table_with_noise,
}

for name, table in with_noise_tables.items():
    print(f"\n{name}")
    display(table)



table_cyl_dependence


,Dependence,avg_ll_mean,avg_ll_std,bic_win_rate,aic_win_rate,ari_mean,time_mean,em_iters_mean
0,ind,-0.00104,0.00272,0.05,12.70,0.17,1.190,1.212
1,strong,0.00438,0.01503,11.65,34.15,0.01,1.174,1.217
2,weak,0.00052,0.00469,7.00,30.30,0.60,1.239,1.424



table_mom_dependence


,L,Dependence,avg_ll_mean,avg_ll_std,bic_win_rate,aic_win_rate,time_mean,em_iters_mean
0,1,ind,0.00783,0.01814,99.50,99.50,2.204,0.794
1,2,ind,0.00287,0.00522,99.42,99.42,1.988,0.618
2,3,ind,0.00221,0.00480,99.08,99.08,1.801,0.402
3,1,strong,0.08413,0.08261,99.25,99.25,2.407,0.952
4,2,strong,0.04066,0.04382,100.00,100.00,2.611,1.096
5,3,strong,0.03388,0.03473,99.92,99.92,2.216,0.644
6,1,weak,0.01903,0.02701,99.83,99.83,2.708,1.155
7,2,weak,0.00915,0.00933,99.58,99.58,2.302,0.826
8,3,weak,0.00700,0.00865,99.67,99.67,2.019,0.511



table_mom_separability


,L,Separability,avg_ll_mean,avg_ll_std,bic_win_rate,aic_win_rate,time_mean,em_iters_mean
0,1,high,0.06601,0.07490,99.72,99.72,2.473,1.026
1,2,high,0.03197,0.03834,100.00,100.00,2.676,1.111
2,3,high,0.02711,0.03038,99.83,99.83,2.346,0.690
3,1,low,0.00799,0.01499,99.33,99.33,2.406,0.908
4,2,low,0.00314,0.00413,99.33,99.33,1.925,0.582
5,3,low,0.00161,0.00310,99.28,99.28,1.678,0.349



table_vs


,Dependence,Separability,avg_ll_mean,avg_ll_std,bic_win_rate,aic_win_rate,hll_win_rate,time_mean,em_iters_mean
0,ind,high,-0.01346,0.01277,0.5,7.5,12.0,0.400,1.108
1,ind,low,-0.00020,0.00307,21.0,57.5,50.0,0.414,1.062
2,strong,high,-0.02041,0.01923,1.5,8.5,9.5,0.203,0.255
3,strong,low,-0.00049,0.00320,23.0,53.5,49.0,0.346,0.817
4,weak,high,-0.01137,0.01031,4.0,5.5,9.0,0.425,1.140
5,weak,low,-0.00013,0.00322,21.0,58.5,52.0,0.397,1.008



ari_table


,Separability,Dependence,Cylindrical,Independent cylindrical,Full MoM,Isolated MoM
0,High,Conditionally independent,10.41,7.22,27.72,29.91
1,High,Weak,21.69,17.91,39.30,30.63
2,High,Strong,60.04,60.80,62.09,34.81
3,Low,Conditionally independent,0.26,0.07,1.03,1.12
4,Low,Weak,1.93,2.10,1.15,1.13
5,Low,Strong,15.04,16.65,3.19,1.27


### Original-minus-noisy table differences

In [20]:
from IPython.display import display

TABLE_ID_COLUMNS = [
    "Separability",
    "Dependence",
    "L",
    "K",
    "Sample",
    "Noise",
    "Experiment",
    "Dimension",
]


def difference_table(original_table, noisy_table, scale_numeric=False):
    label_cols = [
        col
        for col in original_table.columns
        if col in noisy_table.columns and col in TABLE_ID_COLUMNS
    ]
    numeric_cols = [
        col
        for col in original_table.columns
        if col in noisy_table.columns
        and col not in label_cols
        and pd.api.types.is_numeric_dtype(original_table[col])
        and pd.api.types.is_numeric_dtype(noisy_table[col])
    ]

    if label_cols:
        original_indexed = original_table.set_index(label_cols)
        noisy_indexed = noisy_table.set_index(label_cols)
    else:
        original_indexed = original_table.copy()
        noisy_indexed = noisy_table.copy()

    original_values = original_indexed[numeric_cols]
    noisy_values = noisy_indexed[numeric_cols].reindex(original_values.index)
    diff = original_values - noisy_values
    if scale_numeric:
        diff = diff.mul(100)

    if label_cols:
        diff = diff.reset_index()
    diff.columns.name = None
    if scale_numeric:
        diff = round_metric_columns(diff, default=2, skip_columns=label_cols)
    else:
        diff = format_metric_table(diff)

    return diff


original_tables = {
    "table_cyl_dependence": comparison_table(
        df,
        "cylmix",
        "indcylmix",
        match_cols_cyl,
        ["Dependence"],
        COMPARISON_SPECS,
        format_output=False,
    )[["Dependence", "avg_ll_mean", "avg_ll_std", "bic_win_rate", "aic_win_rate", "ari_mean", "time_mean", "em_iters_mean"]],
    "table_mom_dependence": comparison_table(
        df,
        "mom",
        "isomom",
        match_cols_mom,
        ["Dependence", "L"],
        COMPARISON_SPECS,
        format_output=False,
    )[["L", "Dependence", "avg_ll_mean", "avg_ll_std", "bic_win_rate", "aic_win_rate", "time_mean", "em_iters_mean"]],
    "table_mom_separability": comparison_table(
        df,
        "mom",
        "isomom",
        match_cols_mom,
        ["Separability", "L"],
        COMPARISON_SPECS,
        format_output=False,
    )[["L", "Separability", "avg_ll_mean", "avg_ll_std", "bic_win_rate", "aic_win_rate", "time_mean", "em_iters_mean"]],
    "table_vs": comparison_table(
        df,
        "3-Cylindrical Mixture",
        "(3,2)-Two-layer MoM",
        match_cols_vs,
        ["Dependence", "Separability"],
        COMPARISON_SPECS,
        model_col="Model",
        format_output=False,
    )[["Dependence", "Separability", "avg_ll_mean", "avg_ll_std", "bic_win_rate", "aic_win_rate", "hll_win_rate", "time_mean", "em_iters_mean"]],
    "ari_table": ari_recovery_table(df, scale=None, decimals=None),
}

with_noise_tables_raw = {
    "table_cyl_dependence": comparison_table(
        with_noise_df,
        "cylmix",
        "indcylmix",
        match_cols_cyl,
        ["Dependence"],
        COMPARISON_SPECS,
        format_output=False,
    )[["Dependence", "avg_ll_mean", "avg_ll_std", "bic_win_rate", "aic_win_rate", "ari_mean", "time_mean", "em_iters_mean"]],
    "table_mom_dependence": comparison_table(
        with_noise_df,
        "mom",
        "isomom",
        match_cols_mom,
        ["Dependence", "L"],
        COMPARISON_SPECS,
        format_output=False,
    )[["L", "Dependence", "avg_ll_mean", "avg_ll_std", "bic_win_rate", "aic_win_rate", "time_mean", "em_iters_mean"]],
    "table_mom_separability": comparison_table(
        with_noise_df,
        "mom",
        "isomom",
        match_cols_mom,
        ["Separability", "L"],
        COMPARISON_SPECS,
        format_output=False,
    )[["L", "Separability", "avg_ll_mean", "avg_ll_std", "bic_win_rate", "aic_win_rate", "time_mean", "em_iters_mean"]],
    "table_vs": comparison_table(
        with_noise_df,
        "3-Cylindrical Mixture",
        "(3,2)-Two-layer MoM",
        match_cols_vs,
        ["Dependence", "Separability"],
        COMPARISON_SPECS,
        model_col="Model",
        format_output=False,
    )[["Dependence", "Separability", "avg_ll_mean", "avg_ll_std", "bic_win_rate", "aic_win_rate", "hll_win_rate", "time_mean", "em_iters_mean"]],
    "ari_table": ari_recovery_table(with_noise_df, scale=None, decimals=None),
}

difference_tables = {
    name: difference_table(
        original_tables[name],
        with_noise_tables_raw[name],
        scale_numeric=name == "ari_table",
    )
    for name in original_tables
}

for name, table in difference_tables.items():
    print(f"\n{name} (original - noisy)")
    display(table)



table_cyl_dependence (original - noisy)


,Dependence,avg_ll_mean,avg_ll_std,bic_win_rate,aic_win_rate,ari_mean,time_mean,em_iters_mean
0,ind,0.00005,0.00007,0.08,0.65,0.00,0.002,0.001
1,strong,0.00015,0.00039,0.12,0.55,0.01,0.012,0.027
2,weak,0.00009,0.00028,0.67,0.78,0.07,0.003,0.013



table_mom_dependence (original - noisy)


,L,Dependence,avg_ll_mean,avg_ll_std,bic_win_rate,aic_win_rate,time_mean,em_iters_mean
0,1,ind,0.00021,0.00011,-0.08,-0.08,0.015,0.009
1,2,ind,0.00009,0.00018,-0.08,-0.08,0.002,-0.001
2,3,ind,0.00005,0.00012,0.00,0.00,0.001,-0.001
3,1,strong,0.00140,0.00101,0.00,0.00,0.006,0.003
4,2,strong,0.00102,0.00087,0.00,0.00,0.034,0.013
5,3,strong,0.00080,0.00073,0.00,0.00,0.003,-0.010
6,1,weak,0.00021,-0.00001,-0.08,-0.08,0.010,0.010
7,2,weak,0.00028,0.00020,0.04,0.04,-0.004,0.002
8,3,weak,0.00021,0.00019,0.08,0.08,0.005,0.002



table_mom_separability (original - noisy)


,L,Separability,avg_ll_mean,avg_ll_std,bic_win_rate,aic_win_rate,time_mean,em_iters_mean
0,1,high,0.00090,0.00100,0.00,0.00,0.006,-0.003
1,2,high,0.00080,0.00079,0.00,0.00,0.006,0.004
2,3,high,0.00067,0.00064,-0.17,-0.17,-0.008,-0.008
3,1,low,0.00031,0.00016,-0.11,-0.11,0.015,0.018
4,2,low,0.00013,0.00014,-0.03,-0.03,0.014,0.005
5,3,low,0.00004,0.00002,0.22,0.22,0.013,0.003



table_vs (original - noisy)


,Dependence,Separability,avg_ll_mean,avg_ll_std,bic_win_rate,aic_win_rate,hll_win_rate,time_mean,em_iters_mean
0,ind,high,-0.00039,0.00055,0.00,-0.25,1.25,-0.002,-0.019
1,ind,low,0.00006,0.00002,2.25,2.00,0.50,-0.005,-0.006
2,strong,high,-0.00111,0.00008,-0.25,-1.25,-1.25,-0.001,-0.007
3,strong,low,-0.00015,0.00012,1.25,-0.75,0.50,-0.001,0.002
4,weak,high,-0.00017,0.00006,0.00,-0.25,-1.00,-0.007,-0.015
5,weak,low,-0.00000,-0.00003,1.00,-0.75,1.75,-0.001,-0.013



ari_table (original - noisy)


,Separability,Dependence,Cylindrical,Independent cylindrical,Full MoM,Isolated MoM
0,High,Conditionally independent,0.25,0.25,0.06,0.37
1,High,Weak,0.34,0.26,0.52,0.42
2,High,Strong,0.62,0.63,0.84,0.59
3,Low,Conditionally independent,-0.00,-0.00,0.02,0.02
4,Low,Weak,-0.02,0.00,0.03,0.02
5,Low,Strong,0.15,0.13,0.20,0.02
